## **Metodología de Análisis y Pronóstico**

El objetivo de este estudio es desarrollar un modelo predictivo robusto para la serie de tiempo de la población ocupada en las 13 ciudades y áreas metropolitanas de Colombia. Se siguió un enfoque estocástico basado en modelos de espacio de estados (ETS) para capturar las dinámicas de nivel, tendencia y estacionalidad.

### **1. Preparación y Análisis Exploratorio (EDA)**

La serie de tiempo analizada comprende datos mensuales desde enero de 2001 hasta junio de 2019. El proceso inició con la transformación de la variable de fecha a un índice temporal con frecuencia mensual (MS). Se realizó una partición de los datos en dos conjuntos:

* **Entrenamiento (Train):** 210 observaciones (2001-01 a 2018-06).

* **Prueba (Test):** 12 observaciones (2018-07 a 2019-06) para validación cruzada.

### **Marco Teórico: Modelos ETS**

Se utilizó el marco de Suavización Exponencial (ETS: Error-Trend-Seasonality), que permite modelar la evolución de los componentes de la serie mediante ecuaciones de recurrencia. La selección del modelo se basó en la identificación de la naturaleza de los componentes:

* **Nivel ($\ell_t$):** El valor promedio local de la serie.

* **Tendencia ($b_t$):** La dirección de crecimiento o decrecimiento a largo plazo.

* **Estacionalidad ($s_t$):** Los ciclos repetitivos anuales.

### **Evaluación Experimental de Modelos**

Se evaluaron cuatro configuraciones específicas para determinar cuál minimizaba la varianza del error de pronóstico:

* **ETS(A, N, N) - Suavizado Simple:** Utilizado como punto de comparación inicial (baseline), asumiendo que no existe tendencia ni estacionalidad.

* **ETS(M, M, N) - Método de Holt:** Introdujo una tendencia lineal multiplicativa para observar si existía un crecimiento inercial.

* **ETS(A, A, A) - Holt-Winters Aditivo:** Incorporó estacionalidad, asumiendo que los picos y valles de empleo son constantes en términos absolutos año tras año.

* **ETS(A, N, M) - Holt-Winters Multiplicativo sin Tendencia:** Modelo que propone errores aditivos pero una estacionalidad que escala proporcionalmente al nivel de la serie.

### **Criterio de Selección: Validación por RMSE**

La precisión de los modelos se midió utilizando la Raíz del Error Cuadrático Medio (RMSE) en el conjunto de prueba (datos no vistos por el modelo). El RMSE permite cuantificar la magnitud promedio de las desviaciones en las mismas unidades de la serie original (miles de personas):$$RMSE = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2}$$

### **Definición del Modelo Óptimo**

Tras la experimentación, el modelo ETS(A, N, M) resultó superior con un RMSE de 122.14. La superioridad de esta configuración se debe a:

* **Naturaleza Estacional:** El mercado laboral colombiano presenta fluctuaciones proporcionales al volumen de ocupación (multiplicativas), especialmente en temporadas de fin de año.

* **Estacionariedad en Tendencia:** El parámetro de tendencia ($\beta$) resultó ser estadísticamente despreciable, indicando que un modelo sin componente de tendencia es más parsimonioso y evita el sobreajuste (overfitting).

* **Reactividad:** El parámetro de suavizado del nivel ($\alpha \approx 0.605$) indica que el modelo otorga un peso significativo a las observaciones recientes para ajustar el nivel base del pronóstico.

### **Ejecución del Pronóstico**

Finalmente, se reentrenó el modelo óptimo utilizando la totalidad de los datos disponibles (2001-2019) para generar una proyección de 6 meses (Julio - Diciembre 2019). Se calcularon intervalos de confianza al 95% para representar la incertidumbre inherente al horizonte de predicción.



## **Proyecciones del Modelo ETS(A, N, M) para el Segundo Semestre de 2019**



In [1]:
import pandas as pd
import io

# Los datos en formato texto separados por comas (CSV)
datos = """Mes,Pronóstico Puntual,Límite Inferior (95%),Límite Superior (95%)
Julio 2019,10891.60,10680.05,11117.66
Agosto 2019,10861.21,10620.24,11127.44
Septiembre 2019,10936.09,10617.48,11250.85
Octubre 2019,11060.13,10722.57,11393.16
Noviembre 2019,11066.91,10729.46,11442.58
Diciembre 2019,10953.37,10593.82,11327.85"""

# Convertir el texto a un DataFrame de Pandas
df = pd.read_csv(io.StringIO(datos))

# Mostrar la tabla
display(df)

,Mes,Pronóstico Puntual,Límite Inferior (95%),Límite Superior (95%)
0,Julio 2019,10891.60,10680.05,11117.66
1,Agosto 2019,10861.21,10620.24,11127.44
2,Septiembre 2019,10936.09,10617.48,11250.85
3,Octubre 2019,11060.13,10722.57,11393.16
4,Noviembre 2019,11066.91,10729.46,11442.58
5,Diciembre 2019,10953.37,10593.82,11327.85


## **Limitaciones del Modelo y Riesgos del Pronóstico**

A pesar de que el modelo ETS(A, N, M) demostró ser el más preciso en el periodo de validación (minimizando el RMSE), las proyecciones resultantes deben interpretarse teniendo en cuenta las siguientes restricciones metodológicas y contextuales:

### **Naturaleza Univariada y Omisión de Variables Exógenas**

El marco de suavización exponencial asume que toda la información necesaria para predecir el futuro está contenida en el comportamiento pasado de la propia serie temporal. Por lo tanto, el modelo no incorpora variables macroeconómicas causales que son determinantes para el mercado laboral, tales como el crecimiento del Producto Interno Bruto (PIB), la inflación, la Tasa Representativa del Mercado (TRM), o la Inversión Extranjera Directa. Se asume, de forma restrictiva, que el impacto de estas variables ya está codificado en la inercia histórica de la ocupación.

### **Incertidumbre Creciente en el Horizonte de Pronóstico**

Como es inherente a los modelos de series de tiempo, la varianza del error de predicción se acumula a medida que el horizonte se expande. Esto se evidencia claramente en la tabla de proyecciones: los intervalos de confianza al 95% para diciembre de 2019 son considerablemente más amplios que los de julio de 2019. Esto significa que la utilidad del pronóstico puntual pierde fiabilidad técnica a mediano y largo plazo.

### **Vulnerabilidad ante Choques Estructurales (Falta de Flexibilidad)**

El modelo asume que las dinámicas de estacionalidad y nivel se mantendrán constantes durante el segundo semestre de 2019. Sin embargo, este enfoque es "ciego" ante eventos atípicos o cambios estructurales bruscos. Cualquier eventualidad exógena —como la implementación de una nueva reforma laboral, paros nacionales prolongados, o crisis internacionales— invalidaría la proyección, ya que el algoritmo carece de mecanismos para anticipar puntos de quiebre que no tengan precedentes en la data de entrenamiento.

### **Sesgo por Agregación Espacial**

Los datos analizados corresponden al total agregado de ocupados en las 13 principales ciudades y áreas metropolitanas de Colombia. Este nivel de agregación macroeconómica enmascara la heterogeneidad regional. Es posible que el modelo proyecte un crecimiento neto estable, mientras que, internamente, algunas ciudades enfrenten recesiones severas en el empleo que son compensadas por el crecimiento hipertrofiado de ciudades más grandes como Bogotá o Medellín.